# Ejercicio: asistente de documentos con NVIDIA NIM y Gradio

**Tiempo:** 40–45 minutos  
**Grupos:** 3–4 personas

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas
generales. En este ejercicio construiremos un asistente que responda preguntas
sobre un documento específico: **“Attention Is All You Need”**
(Vaswani et al., 2017).

Inyectar un documento completo en el contexto permite construir un primer
prototipo de *document question answering*. Esto ayuda a entender la motivación
de **RAG**, aunque todavía no es RAG: aquí no existe recuperación de fragmentos.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:

1. Cargar o leer el PDF del paper.
2. Hacer preguntas sobre su contenido.
3. Obtener respuestas basadas exclusivamente en el documento.

## Lo que van a aprender

- Extracción de texto de PDF con `pypdf`.
- Uso de contexto externo en un `system` prompt.
- Consumo de NVIDIA NIM mediante una API compatible con OpenAI.
- Streaming e historial con Gradio.
- Limitaciones de este enfoque y motivación de RAG.

> **Nota de la solución:** en esta solución usamos **Groq** en vez de NVIDIA NIM. Groq también ofrece una API compatible con OpenAI, así que el código es el mismo cambiando `base_url`, la variable `GROQ_API_KEY` y el nombre del modelo. Crea tu llave en [console.groq.com/keys](https://console.groq.com/keys).

## Paso 0: instalación y configuración

### 0.1 Instala las dependencias

- `pypdf`: extracción del texto del PDF.
- `gradio`: interfaz web.
- `openai`: cliente compatible con Groq (y con NVIDIA NIM).
- `python-dotenv`: lectura de variables de entorno.

```bash
pip install -q --upgrade pypdf gradio openai python-dotenv
```

In [1]:
# Instalamos las dependencias (solo hace falta la primera vez)
%pip install -q --upgrade pypdf gradio openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


### 0.2 Descarga el paper

Descarga el PDF abierto desde:

```text
https://arxiv.org/pdf/1706.03762
```

Guárdalo junto al notebook como `attention_is_all_you_need.pdf` (en esta carpeta ya está incluido).


> **Límite importante:** en el plan gratuito, los modelos de Groq aceptan máximo **8.000 tokens por minuto**. El paper completo ocupa ~10.400 tokens, así que lo recortamos quitando las referencias y las páginas de visualizaciones del final (ver Paso 1).

## Paso 1: extracción de texto del PDF

Lee el PDF y extrae su contenido como texto plano.

**Instrucciones:**

1. Importa `PdfReader` desde `pypdf`.
2. Crea `extract_text_from_pdf(pdf_path)`.
3. Abre el PDF e itera sobre `reader.pages`.
4. Concatena el resultado de `page.extract_text()`; contempla páginas sin texto.
5. Retorna el texto completo.
6. Prueba la función e imprime los primeros 500 caracteres.

> **Pista:** usa `page.extract_text() or ""` para evitar valores `None`.

In [1]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    # Abrimos el PDF
    reader = PdfReader(pdf_path)
    text = ""
    # Recorremos las páginas y concatenamos su texto
    for page in reader.pages:
        # Si una página no tiene texto, extract_text() puede devolver None
        text += (page.extract_text() or "") + "\n"
    return text

PDF_PATH = "attention_is_all_you_need.pdf"

document_text = extract_text_from_pdf(PDF_PATH)
print(document_text[:500])
print("...")
print(f"Caracteres extraídos: {len(document_text):,}")


def recortar_documento(text):
    # El plan gratuito de Groq permite máximo ~8.000 tokens por minuto
    # y el paper completo ocupa ~10.400 tokens, así que no cabe.
    # Quitamos todo desde "References" (bibliografía y páginas de
    # visualizaciones de atención), que no se necesita para responder.
    corte = text.find("References")
    if corte == -1:
        return text
    return text[:corte]

document_text = recortar_documento(document_text)
print(f"Caracteres después de recortar: {len(document_text):,}")

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 
...
Caracteres extraídos: 39,600
Caracteres después de recortar: 30,128


## Paso 2: inicialización del cliente (Groq)

**Instrucciones (adaptadas a Groq):**

1. Importa `os` y `OpenAI`.
2. Define la llave `GROQ_API_KEY`.
3. Verifica que la llave esté disponible sin imprimirla.
4. Inicializa `OpenAI` con:
   - `base_url="https://api.groq.com/openai/v1"`
   - `api_key=GROQ_API_KEY`
5. Define el modelo:

```python
MODELO = "openai/gpt-oss-120b"
```

In [2]:
import os
from openai import OpenAI

# Llave de Groq (ejercicio académico: la dejamos directamente en el notebook).
# Si existe la variable de entorno GROQ_API_KEY, se usa esa.
GROQ_API_KEY = os.getenv("GROQ_API_KEY") or "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"

# Verificamos que la llave exista, pero SIN imprimirla
print("✓ Llave de Groq configurada" if GROQ_API_KEY else "⚠ Falta la llave de Groq")

# Groq ofrece una API compatible con OpenAI:
# solo cambiamos la URL base y la llave.
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY,
)

# Modelo disponible en Groq (lista completa: https://console.groq.com/docs/models)
MODELO = "openai/gpt-oss-120b"

✓ Llave de Groq configurada


## Paso 3: función de chat con el documento

Construye un `system` prompt que incluya el texto del paper y obligue al modelo
a fundamentar sus respuestas en él.

**Instrucciones:**

1. Crea `build_system_prompt(document_text)` para:
   - definir al asistente como experto en el paper;
   - incluir el documento entre delimitadores claros;
   - exigir respuestas basadas únicamente en el documento;
   - indicar que responda “No encuentro esa información en el documento” cuando corresponda.

2. Crea `history_to_messages(history)` para convertir el historial de Gradio a
   mensajes con roles `user` y `assistant`.

3. Crea `chat_con_documento(message, history, document_text)` para:
   - iniciar `messages` con el `system` prompt;
   - agregar el historial y el mensaje actual;
   - llamar `client.chat.completions.create(..., stream=True)`;
   - acumular `chunk.choices[0].delta.content`;
   - usar `yield` para entregar la respuesta progresivamente.

> **Pista:** utiliza `temperature=1.0` y `top_p=0.95`, valores recomendados para
> Nemotron 3 Super.

In [3]:
def build_system_prompt(document_text):
    # El system prompt define el rol, incluye el documento y pone las reglas
    return (
        "Eres un asistente experto en el paper 'Attention Is All You Need' (Vaswani et al., 2017).\n"
        "Responde ÚNICAMENTE con base en el documento que aparece entre las marcas "
        "<<<DOCUMENTO>>> y <<<FIN_DOCUMENTO>>>. No uses conocimiento externo.\n"
        "Si la respuesta no está en el documento, responde exactamente: "
        "\"No encuentro esa información en el documento\".\n"
        "Responde en español y de forma clara.\n\n"
        "<<<DOCUMENTO>>>\n"
        f"{document_text}\n"
        "<<<FIN_DOCUMENTO>>>"
    )


def history_to_messages(history):
    # Convierte el historial de Gradio en mensajes para la API.
    messages = []
    for item in history:
        # Formato nuevo de Gradio: lista de diccionarios {"role": ..., "content": ...}
        if isinstance(item, dict):
            if item.get("role") in ("user", "assistant"):
                messages.append({"role": item["role"], "content": str(item["content"])})
        # Formato antiguo: lista de pares [usuario, asistente]
        else:
            user_msg, bot_msg = item
            if user_msg:
                messages.append({"role": "user", "content": user_msg})
            if bot_msg:
                messages.append({"role": "assistant", "content": bot_msg})
    return messages


def chat_con_documento(message, history, document_text):
    # 1. Empezamos con el system prompt que contiene el documento
    messages = [{"role": "system", "content": build_system_prompt(document_text)}]
    # 2. Agregamos la conversación anterior y la pregunta actual
    # Solo enviamos los últimos 4 mensajes del historial para no pasarnos
    # del límite de tokens por minuto (el documento ya ocupa casi todo).
    messages += history_to_messages(history)[-4:]
    messages.append({"role": "user", "content": message})

    # 3. Llamamos al modelo con streaming
    stream = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=0.3,   # temperatura baja: respuestas más fieles al documento
        top_p=0.95,
        max_tokens=800,
        reasoning_effort="low",   # gpt-oss "piensa" antes de responder; "low" lo hace rápido
        stream=True,
    )

    # 4. Vamos acumulando los fragmentos y los entregamos con yield
    respuesta = ""
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            respuesta += chunk.choices[0].delta.content
            yield respuesta

## Paso 4: interfaz Gradio

Construye la interfaz con `gr.ChatInterface`.

**Instrucciones:**

1. Extrae el texto del PDF con la función del paso 1.
2. Muestra cuántas páginas y caracteres se extrajeron.
3. Configura `gr.ChatInterface` con:
   - `fn=chat_con_documento`;
   - un título y una descripción;
   - un `gr.Textbox` oculto dentro de `additional_inputs` para enviar el documento;
   - al menos tres preguntas de ejemplo.

```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz:

```python
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
```

In [4]:
import gradio as gr
from pypdf import PdfReader

# 1. Extraemos el texto del PDF (y lo recortamos para que quepa en el límite de Groq)
document_text = recortar_documento(extract_text_from_pdf(PDF_PATH))

# 2. Mostramos cuántas páginas y caracteres se extrajeron
num_paginas = len(PdfReader(PDF_PATH).pages)
print(f"Páginas: {num_paginas}")
print(f"Caracteres: {len(document_text):,}")

# 3. Interfaz de chat
demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="Asistente: Attention Is All You Need",
    description="Haz preguntas sobre el paper. El asistente responde solo con información del documento.",
    # Textbox oculto: así le pasamos el documento a la función
    additional_inputs=[gr.Textbox(value=document_text, visible=False)],
    examples=[
        ["¿Cuál es la arquitectura principal propuesta en el paper?", document_text],
        ["¿Qué es el mecanismo de atención?", document_text],
        ["¿Cuántas capas tiene el encoder del modelo base?", document_text],
    ],
)

# 4. Lanzamos la interfaz (abre http://localhost:8080)
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)

c:\Users\Tomas\OneDrive\Desktop\Archivos y programa EAFIT\IA\Solucion Talleres\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Páginas: 15
Caracteres: 30,128
* Running on local URL:  http://0.0.0.0:8080
* To create a public link, set `share=True` in `launch()`.


## Paso 5: Prueba y reflexión
 
Una vez que la interfaz esté funcionando, prueba estas preguntas:
 
1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*
 
Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*
 
Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?
 

### Prueba sin interfaz

Esta celda hace las 6 preguntas directamente con la función (sin Gradio). Solo funciona si la llave de Groq está configurada. Esperamos unos segundos entre preguntas porque el plan gratuito de Groq limita los tokens por minuto y cada pregunta envía el paper completo (~10.000 tokens).

In [5]:
preguntas = [
    "¿Cuál es la arquitectura principal propuesta en el paper?",
    "¿Qué es el mecanismo de atención?",
    "¿Cuántas capas tiene el encoder del modelo base?",
    "¿Quiénes son los autores del paper?",
    "¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?",
    "¿Qué es GPT-4?",   # pregunta trampa
]

import time

if GROQ_API_KEY:
    for p in preguntas:
        respuesta_final = ""
        try:
            for parcial in chat_con_documento(p, [], document_text):
                respuesta_final = parcial      # nos quedamos con la respuesta completa
        except Exception as e:
            respuesta_final = f"(Error al llamar la API: {e})"
        print("Pregunta:", p)
        print("Respuesta:", respuesta_final)
        print("-" * 80)
        time.sleep(65)   # respetamos el límite de tokens por minuto del plan gratuito
else:
    print("Sin GROQ_API_KEY: configura el archivo .env para probar las preguntas.")

Pregunta: ¿Cuál es la arquitectura principal propuesta en el paper?
Respuesta: El paper propone la arquitectura del **Transformer**, un modelo de transducción de secuencias basado únicamente en mecanismos de atención, sin recurrencia ni convoluciones.
--------------------------------------------------------------------------------
Pregunta: ¿Qué es el mecanismo de atención?
Respuesta: El mecanismo de atención es una función que, dado una consulta (query) y un conjunto de pares clave‑valor (key‑value), produce una salida calculada como una suma ponderada de los valores. Los pesos de cada valor se determinan mediante una función de compatibilidad entre la consulta y la clave correspondiente. En el Transformer se implementa como “Scaled Dot‑Product Attention”, donde se calculan los productos punto entre consultas y claves, se escalan por \(1/\sqrt{d_k}\), se aplica una softmax para obtener los pesos y luego se multiplican por los valores. Además, el modelo utiliza “Multi‑Head Attention”, 

### Qué esperamos (según el paper)

| Pregunta | Respuesta esperada en el documento |
|---|---|
| 1. Arquitectura principal | El **Transformer**: un modelo encoder-decoder basado solo en mecanismos de atención, sin recurrencia ni convoluciones. |
| 2. Mecanismo de atención | Una función que toma una *query* y un conjunto de pares *key-value* y devuelve una suma ponderada de los *values*; los pesos salen de la compatibilidad entre la *query* y cada *key*. En el paper usan *Scaled Dot-Product Attention*: $\text{softmax}(QK^T/\sqrt{d_k})V$, y *Multi-Head Attention*. |
| 3. Capas del encoder | $N = 6$ capas idénticas. |
| 4. Autores | Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser e Illia Polosukhin. |
| 5. WMT 2014 EN-DE | El Transformer (big) obtiene **28.4 BLEU**, nuevo estado del arte; el modelo base obtiene 27.3. |
| 6. ¿Qué es GPT-4? (trampa) | No aparece en el paper (es de 2017). Si el modelo sigue la instrucción, debe responder *"No encuentro esa información en el documento"*. |

**Resultado obtenido con Groq (`openai/gpt-oss-120b`):** las 5 preguntas sobre el paper se respondieron correctamente (Transformer, *Scaled Dot-Product* y *Multi-Head Attention*, 6 capas, los 8 autores y 28.4 BLEU). En la pregunta trampa sobre GPT-4 el modelo **respetó la instrucción** y respondió *"No encuentro esa información en el documento"*, aunque sí conoce GPT-4 por su entrenamiento. Esto muestra que el `system` prompt funciona, pero no es una garantía: con otras preguntas o con otro modelo podría "escaparse" y usar conocimiento general, por eso siempre conviene verificar las respuestas.

## Paso 6 (adicional): mejora el sistema

Si terminan antes, implementen una mejora:

**A) Medición del contexto**  
Reporta la cantidad de caracteres o estima los tokens del documento. Explica
por qué caracteres y tokens no son equivalentes.

**B) Subida dinámica de PDF**  
Agrega `gr.File` para cargar cualquier PDF y extrae el texto dentro del flujo de
la aplicación.

**C) Evidencia y páginas**  
Conserva separadores de página al extraer el PDF y exige que cada respuesta cite
la página que contiene la evidencia.

**D) Primer paso hacia RAG**  
Divide el documento en fragmentos y selecciona los más relevantes mediante una
búsqueda simple por palabras antes de llamar al modelo.

### Mejora A — Medición del contexto

Calculamos cuántos caracteres y palabras tiene el documento y estimamos los tokens. Una regla práctica para texto en inglés es **≈ 4 caracteres por token**.

In [6]:
num_caracteres = len(document_text)
num_palabras = len(document_text.split())
tokens_estimados = num_caracteres // 4   # aproximación: ~4 caracteres por token

print(f"Caracteres       : {num_caracteres:,}")
print(f"Palabras         : {num_palabras:,}")
print(f"Tokens estimados : {tokens_estimados:,}")

Caracteres       : 30,128
Palabras         : 4,703
Tokens estimados : 7,532


**Resultado:** el documento recortado tiene ≈ 30.100 caracteres, ≈ 4.700 palabras y ≈ 7.500 tokens estimados (el paper completo tenía ≈ 39.600 caracteres ≈ 10.400 tokens según Groq, por eso hubo que recortarlo para no pasar el límite de 8.000 tokens por minuto).

**¿Por qué caracteres y tokens no son equivalentes?** Los modelos no leen caracteres sino **tokens**, que son pedazos de texto definidos por el tokenizador (palabras completas, partes de palabra, signos de puntuación o espacios). Una palabra común en inglés puede ser un solo token, mientras que palabras raras, números, fórmulas o texto en otros idiomas se parten en varios tokens. Por eso la relación caracteres/tokens cambia según el texto y el tokenizador, y "4 caracteres por token" es solo una estimación. El límite de la ventana de contexto se mide en tokens.

## Reflexión final

Discutan en grupo:

1. ¿Cuál es la limitación de enviar siempre el documento completo?
2. ¿Por qué este ejercicio todavía no implementa RAG?
3. ¿Cómo comprobarían que la respuesta proviene del documento y no del
   conocimiento previo del modelo?
4. ¿Qué riesgos aparecen al insertar directamente documentos de usuarios en el
   prompt?

### Respuestas

**1.** El documento entero se envía en **cada** pregunta, así que: (a) consume muchos tokens, lo que hace cada llamada más lenta y más costosa; (b) si el documento es más grande que la ventana de contexto del modelo, simplemente no cabe; y (c) con textos muy largos el modelo puede "perderse" y prestar menos atención a la parte relevante. Funciona con un paper de 15 páginas, pero no con un libro o con cientos de documentos.

**2.** Porque no hay **recuperación (retrieval)**: no dividimos el documento en fragmentos, no calculamos embeddings ni buscamos los fragmentos más relevantes para cada pregunta. Solo pegamos todo el texto en el prompt. RAG = *Retrieval-Augmented Generation*: primero se recupera lo relevante y solo eso se le da al modelo.

**3.** Algunas formas:
- Pedir que cite la **página o la frase exacta** del documento y verificar que esa cita exista en el texto.
- Hacer **preguntas trampa** cuya respuesta no esté en el documento (como GPT-4) y revisar que diga "No encuentro esa información".
- Comparar con la respuesta del modelo **sin** el documento: si es igual, probablemente viene de su conocimiento previo.
- Modificar un dato del documento (por ejemplo, cambiar 6 capas por 8) y ver si el modelo responde con el dato modificado.

**4.** Riesgos principales:
- **Prompt injection:** el documento puede traer instrucciones ocultas ("ignora las reglas anteriores y ...") que el modelo podría obedecer.
- **Privacidad:** los documentos pueden tener datos personales o confidenciales que se envían a un servicio externo.
- **Costo / abuso:** documentos enormes pueden saturar la ventana de contexto o generar costos altos.
- **Desinformación:** si el documento tiene información falsa, el modelo la repetirá como si fuera cierta.